In [ ]:
#Setup
import time
import os
import asyncio
import datetime
import json
import re
from google import genai
from google.genai import errors, types
from google.colab import drive, userdata
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type


drive.mount('/content/drive')
API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=API_KEY)

OUTPUT_DIR = "V:\research_agents\mad_experiments\exp_4"
os.makedirs(OUTPUT_DIR, exist_ok=True)
ACTIVE_MODEL = "gemini-2.5-flash"

# If 2.5-flash runs out, we cascade down through the best available models
FALLBACK_MODELS = [
    "gemini-2.5-pro",
    "gemini-2.0-flash",
    "gemini-2.0-flash-001",
    "gemini-2.0-flash-lite-001",
    "gemini-2.0-flash-lite",
    "gemini-flash-latest",
    "gemini-flash-lite-latest",
    "gemini-2.5-flash-lite",
    "gemini-3-pro-preview",
    "gemini-3-flash-preview",
    "gemini-2.5-flash",
    "gemini-2.5-flash-native-audio-preview-12-2025",
    "gemini-2.5-flash-lite",
    "gemini-2.0-flash",
    "gemini-2.0-flash-lite",
    "gemini-3.1-flash-lite",  # Fallback 1: 15 RPM / 500 RPD
    "gemma-3-27b",            # Fallback 2: 30 RPM / 14.4K RPD (Massive limits!)
    "gemini-2.5-flash-lite"   # Fallback 3: 10 RPM / 20 RPD (Last resort)
]
# add gemini 3 flash and 3 flash live if I exhaust this too

@retry(
    stop=stop_after_attempt(12),
    wait=wait_exponential(multiplier=10, min=30, max=300),
    retry=retry_if_exception_type((errors.ClientError, Exception))
)
def generate_safe_content(system_prompt: str, user_prompt: str, role: str, files=None):
    global ACTIVE_MODEL, FALLBACK_MODELS
    # MOVE THE PRINT STATEMENT HERE so it fires on every retry!
    print(f"[{role}] Generating response using {ACTIVE_MODEL}...")

    try:
        content_list = [user_prompt]
        if files: content_list.extend(files)

        response = client.models.generate_content(
            model=ACTIVE_MODEL,
            contents=content_list,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt, temperature=0.4
            )
        )
        return response.text
    except Exception as e:
        msg = str(e).upper()
        if "QUOTA" in msg and ("DAY" in msg or "EXCEEDED" in msg):
            if FALLBACK_MODELS:
                print(f"\n[🚨 QUOTA] {ACTIVE_MODEL} exhausted. Rerouting to {FALLBACK_MODELS[0]}...")
                ACTIVE_MODEL = FALLBACK_MODELS.pop(0)
                raise e
            else:
                raise Exception("All fallback models exhausted for the day.")
        elif "429" in msg or "503" in msg or "UNAVAILABLE" in msg:
            print(f"  [API Traffic] Delaying request on {ACTIVE_MODEL}...")
            raise e
        else:
            raise e

async def call_llm_serial(system_prompt: str, user_prompt: str, role: str, files=None) -> str:
    # We remove the print statement from here, but pass the role down
    return await asyncio.to_thread(generate_safe_content, system_prompt, user_prompt, role, files)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
from google import genai

# Fetch your API key safely (handles both Colab and local environments)
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY') # Make sure this matches your secret name!
except ImportError:
    api_key = os.environ.get("GEMINI_API_KEY")

# Initialize the client
client = genai.Client(api_key=api_key)

print("Currently Active Models for your API Key:")
# List all available models
for m in client.models.list():
    print(f" - {m.name}")

Currently Active Models for your API Key:
 - models/gemini-2.5-flash
 - models/gemini-2.5-pro
 - models/gemini-2.0-flash
 - models/gemini-2.0-flash-001
 - models/gemini-2.0-flash-lite-001
 - models/gemini-2.0-flash-lite
 - models/gemini-2.5-flash-preview-tts
 - models/gemini-2.5-pro-preview-tts
 - models/gemma-3-1b-it
 - models/gemma-3-4b-it
 - models/gemma-3-12b-it
 - models/gemma-3-27b-it
 - models/gemma-3n-e4b-it
 - models/gemma-3n-e2b-it
 - models/gemma-4-26b-a4b-it
 - models/gemma-4-31b-it
 - models/gemini-flash-latest
 - models/gemini-flash-lite-latest
 - models/gemini-pro-latest
 - models/gemini-2.5-flash-lite
 - models/gemini-2.5-flash-image
 - models/gemini-3-pro-preview
 - models/gemini-3-flash-preview
 - models/gemini-3.1-pro-preview
 - models/gemini-3.1-pro-preview-customtools
 - models/gemini-3.1-flash-lite-preview
 - models/gemini-3-pro-image-preview
 - models/nano-banana-pro-preview
 - models/gemini-3.1-flash-image-preview
 - models/lyria-3-clip-preview
 - models/lyria-3

In [ ]:
#Ensure PDF is being read correctly
paper1 = client.files.upload(file='/content/drive/MyDrive/SENIOR_YEAR/FED/MadTest1.pdf')

def wait_for_files_active(files):
    print("Verifying file status...")
    for f in files:
        file_status = client.files.get(name=f.name)
        while file_status.state.name == "PROCESSING":
            time.sleep(2)
            file_status = client.files.get(name=f.name)
        if file_status.state.name == "FAILED": raise Exception("File failed to process.")
    print(f"  ✓ {files[0].display_name} is ACTIVE.")

wait_for_files_active([paper1])

# --- PHASE 1: SANITY CHECK ---
#print("\n--- RUNNING PDF EXTRACTION SANITY CHECK ---")
#sanity_prompt = """
#Please extract and return ONLY the following three items from the document:
#1. The EXACT first 2 sentences of the Abstract.
#2. The exact title/caption of Table 1 (or Figure 1 if no tables exist).
#3. The exact final concluding sentence of the paper (before the references/#appendix).
#4. A summary of the paper's purpose, findings, and significance.
#"""
#sanity_result = generate_safe_content("You are a precise data extraction tool.#", sanity_prompt, [paper1])
#print(sanity_result)
#print("\n[VERIFY THIS OUTPUT AGAINST YOUR PDF BEFORE PROCEEDING]")

Verifying file status...
  ✓ None is ACTIVE.


In [ ]:
#Initializing the environment and setting up personas
# Note the {N} variable for Phase 4 experimentation
SELECTION_PROMPT = """
You are the Chief Editor of an economics journal. You must select exactly {N} expert personas to review the provided paper.
The available personas are:
1. "Theorist": Rigorous mathematical logic, logically airtight explanations and proof

2. "Econometrician": Compelling causal inference, well-defined and constructed identification and estimation strategies, robust interpretation of results without overclaiming

3. "ML_Expert": Fundamental machine learning models, traditional ML, neural architecture, modern ML, well-justified hyperparameter decisions

4. "Data_Scientist": Data cleaning, processing, engineering, manipulation visualization, analysis, interpretation, cleanliness and interpretability of data manipulations

5. "CS_Expert": Sophistication in algorithm creation, computability, complexity, specification/implementation duality, recursion, fixpoint, scale, function/ data duality, static/dynamic duality, modeling, interaction

6. "Historian": Literary history, subject-matter-corpus-specific context, accurate framing of the research narrative

7. "Visionary": Potential for paradigm shifts; broad intellectual novelty

8. "Policymaker": Real-world applicability, regulatory use, welfare implications, political and policy-making relevance

9. "Ethicist": Moral hazard, adverse selection, selection bias, overrepresented/underrepresented literature, privacy, consent, following a standard of conduct, fairness, accountability, adherence to moral and social values

10. "Perspective": Distributional consequences, algorithmic fairness, impact of research on marginalized groups and coverage of marginalized groups within research, racism, sexism, homophobia, transphobia, etc.

### OBJECTIVE

Select the {N} most crucial personas for reviewing THIS SPECIFIC PAPER. Assign them weights based on relevance. Weights must sum exactly to 1.0.

OUTPUT FORMAT (STRICT JSON):
{{
  "selected_personas": ["Persona1", "Persona2", ...],
  "weights": {{"Persona1": 0.5, "Persona2": 0.5, ...}},
  "justification": "1 sentence explaining the choice."
}}
"""

SYSTEM_PROMPTS = {
    "Theorist": "ROLE: Pure Economic Theorist. Focus ONLY on mathematical logic, proofs, and the soundness of derivations.",

    "Econometrician": "ROLE: Econometrician. Focus ONLY on causal inference, endogeneity, identification strategies, and the robustness of results and interpretation.",

    "ML_Expert": "ROLE: Machine Learning/AI Expert. Focus ONLY on the model architecture decisions, structure, and execution (e.g., transformers, dimensionality reduction algorithms), hyperparameter tuning, train/test validity, model explanation, interpretability, and relevance; keep Occam's Razor in mind.",

    "Data_Scientist": "ROLE: Data Science Expert. Focus ONLY on the data pipeline: data cleaning decisions, feature engineering, exploratory data analysis (EDA), data leakage, preprocessing biases, and potential mistakes.",

    "CS_Expert": "ROLE: Computer Science Expert. Focus ONLY on algorithm creation if it is not SOLEY ML, computational complexity and efficiency, memory efficiency, hardware constraints.",

    "Historian": "ROLE: Historian of Thought. Focus ONLY on literature lineage and how well the author represents that lineage, characterizes their work within the lineage, and contributes to the lineage.",

    "Visionary": "ROLE: Visionary. Focus ONLY on paradigm-shifting potential. Does this challenge existing frameworks, or is it merely incremental? View economics from both an insider and outsider perspective when answering these questions.",

    "Policymaker": "ROLE: Policymaker. Focus ONLY on real-world utility. Can a central bank or regulator use this? Are there welfare implications, and are they actionable?",

    "Ethicist": "ROLE: Ethicist. Focus ONLY on the adherence of this premise and construction on moral and social values, privacy, consent, fairness, accountability,philosophical implications of the research.",

    "Perspective": "ROLE: Perspective/DEI Expert. Focus ONLY on distributional consequences. Does this dataset contain inherent biases? Does the algorithm lack fairness? How does this impact marginalized groups? Are marginalized groups represented? "
}

# Add strict formatting rules to all personas dynamically
for role in SYSTEM_PROMPTS:
    SYSTEM_PROMPTS[role] += """
    ### OBJECTIVE & OUTPUT FORMAT
    - **Domain Audit**: [Provide your critique strictly within your assigned domain]
    - **Proportional Error Analysis**: [Contextualize errors: are they fatal to the paper, or minor typos?]
    - **Uncertainty Disclosure**: [Explicitly state what you are unsure about regarding the paper]
    - **Source Evidence**: [MANDATORY: verbatim quotes/equations/tables]
    - **Verdict**: [PASS/REVISE/FAIL]
    """

DEBATE_PROMPTS = {
    "Round_2A_Cross_Examination": """
    ### CONTEXT: You are the {role}. Read your peers' Round 1 evaluations:
    {peer_reports}

    ### OBJECTIVE
    Engage in cross-domain examination. Employ the **Principle of Charity**: assume your peers' points are valid until proven otherwise, but push back vigorously (yet respectfully) if they violate the truth of your domain.

    ### OUTPUT FORMAT
    - **Insights Absorbed**: [How their views change your perspective]
    - **Constructive Pushback**: [Clashes between your domain and theirs]
    - **Clarification Requests**: [Ask 1 specific question to each peer]
    """,

    "Round_2B_Direct_Examination": """
    ### CONTEXT: You are the {role}. Here are the questions directed at you:
    {r2a_transcript}

    ### OBJECTIVE
    Answer the questions directed at you directly using textual evidence. Do not dodge.

    ### OUTPUT FORMAT
    - **Direct Responses**: [Answer each peer's question]
    - **Concession or Defense**: [Explicitly state if you concede a flaw or defend your ground]
    """,

    "Round_2C_Final_Amendment": """
    ### CONTEXT: Full Debate Transcript:
    {debate_transcript}

    ### OBJECTIVE
    Submit your Final Amended Report.

    ### OUTPUT FORMAT
    - **Insights Absorbed**: [How the debate changed your evaluation]
    - **Final Verdict**: [PASS / REVISE / FAIL]
    - **Final Rationale**: [3-sentence justification incorporating debate context]
    """,

    "Round_3_Editor": """
    ### CONTEXT: You are the Senior Editor. Calculate the endogenous weighted consensus of the panel and write the final decision letter.

    ### PANEL CONTEXT & WEIGHTS
    {weights_json}

    ### AMENDED REPORTS
    {final_reports_text}

    ### THE ENDOGENOUS WEIGHTING SYSTEM (STRICT)
    1. Assign values to verdicts: PASS = 1.0, REVISE = 0.5, FAIL = 0.0.
    2. Multiply each persona's value by their assigned weight.
    3. Sum the weighted values to get the Final Consensus Score (out of 1.0).
    4. Thresholds: > 0.75 : ACCEPT | 0.40 - 0.75 : REJECT AND RESUBMIT | < 0.40 : REJECT

    ### OUTPUT FORMAT
    - **Weight Calculation**: [Show math]
    - **Debate Synthesis**: [2-3 sentences summarizing alignment]
    - **Final Decision**: [ACCEPT / REJECT AND RESUBMIT / REJECT]
    - **Official Referee Report**: [Synthesized letter drawing ONLY from panel findings]
    """
}

In [ ]:

# --- ORCHESTRATION PIPELINE FOR DYNAMIC 'N' ---
async def run_round_0_selection(paper_file, N) -> dict:
    print(f"\n--- STARTING ROUND 0: ENDOGENOUS SELECTION (N={N}) ---")
    prompt = SELECTION_PROMPT.format(N=N)
    response = await call_llm_serial("You are an AI routing agent.", prompt, "Editor_Selector", files=[paper_file])

    try:
        json_match = re.search(r"\{.*\}", response, re.DOTALL)
        selection_data = json.loads(json_match.group())
        personas = selection_data.get("selected_personas", [])
        if len(personas) != N:
            print(f"[WARNING] Requested {N} personas, but LLM returned {len(personas)}.")
        print(f"[SYSTEM] Selected Panel: {personas}")
        return selection_data
    except Exception as e:
        print(f"[ERROR] JSON parse failed. Defaulting to first {N} options. Error: {e}")
        # Dynamic fallback based on N
        default_panel = list(SYSTEM_PROMPTS.keys())[:N]
        eq_weight = round(1.0 / N, 2)
        return {
            "selected_personas": default_panel,
            "weights": {p: eq_weight for p in default_panel}
        }

async def run_round_1_serial(active_personas, paper_file) -> dict:
    print("\n--- STARTING ROUND 1: INDEPENDENT EVALUATION ---")
    reports = {}
    for role in active_personas:
        reports[role] = await call_llm_serial(SYSTEM_PROMPTS[role], "Evaluate this paper based on your role.", role, files=[paper_file])
        await asyncio.sleep(20) # Gentle cooldown between personas
    return reports

async def run_round_2a_serial(r1_reports, active_personas, paper_file) -> dict:
    print("\n--- STARTING ROUND 2A: CROSS-EXAMINATION ---")
    reports = {}
    for role in active_personas:
        peers = [p for p in active_personas if p != role]
        peer_reports_text = "\n".join([f"--- {p} Report ---\n{r1_reports[p]}\n" for p in peers])

        prompt_2a = DEBATE_PROMPTS["Round_2A_Cross_Examination"].format(
            role=role, peer_reports=peer_reports_text
        )
        reports[role] = await call_llm_serial(SYSTEM_PROMPTS[role], prompt_2a, role, files=[paper_file])
        await asyncio.sleep(20)
    return reports

async def run_round_2b_serial(r2a_reports, active_personas, paper_file) -> dict:
    print("\n--- STARTING ROUND 2B: ANSWERING CLARIFICATIONS ---")
    r2a_transcript = "\n".join([f"[{r} CROSS-EXAMINATION]:\n{text}\n" for r, text in r2a_reports.items()])

    reports = {}
    for role in active_personas:
        prompt_2b = DEBATE_PROMPTS["Round_2B_Direct_Examination"].format(
            role=role, r2a_transcript=r2a_transcript
        )
        reports[role] = await call_llm_serial(SYSTEM_PROMPTS[role], prompt_2b, role, files=[paper_file])
        await asyncio.sleep(20)
    return reports

async def run_round_2c_serial(r1_reports, r2a_reports, r2b_reports, active_personas, paper_file) -> dict:
    print("\n--- STARTING ROUND 2C: FINAL AMENDMENTS ---")
    transcript = "ROUND 1 REPORTS:\n" + "\n".join([f"[{r}]:\n{t}" for r, t in r1_reports.items()])
    transcript += "\nCROSS-EXAMINATION:\n" + "\n".join([f"[{r}]:\n{t}" for r, t in r2a_reports.items()])
    transcript += "\nANSWERS:\n" + "\n".join([f"[{r}]:\n{t}" for r, t in r2b_reports.items()])

    reports = {}
    for role in active_personas:
        prompt_2c = DEBATE_PROMPTS["Round_2C_Final_Amendment"].format(role=role, debate_transcript=transcript)
        reports[role] = await call_llm_serial(SYSTEM_PROMPTS[role], prompt_2c, role, files=[paper_file])
        await asyncio.sleep(20)
    return reports

async def run_round_3(r2c_reports, selection_data) -> str:
    print("\n--- STARTING ROUND 3: EDITOR DECISION ---")
    weights_json = json.dumps(selection_data["weights"], indent=2)
    final_reports_text = "\n".join([f"--- {r.upper()} FINAL REPORT ---\n{t}" for r, t in r2c_reports.items()])

    prompt_3 = DEBATE_PROMPTS["Round_3_Editor"].format(weights_json=weights_json, final_reports_text=final_reports_text)
    return await call_llm_serial("You are the Senior Editor.", prompt_3, "Editor")

async def execute_experiment(paper_file, N_personas):
    """Wraps the pipeline for Phase 4 experiments."""
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    exp_folder = os.path.join(OUTPUT_DIR, f"Experiment_N{N_personas}")
    os.makedirs(exp_folder, exist_ok=True)

    log_file_path = os.path.join(exp_folder, f"Review_N{N_personas}_{timestamp}.txt")
    print(f"\n[SYSTEM] Saving outputs to: {log_file_path}")

    def log_stage(stage_title, reports=None, single_text=None):
        with open(log_file_path, "a", encoding="utf-8") as f:
            f.write(f"\n{'='*60}\n{stage_title}\n{'='*60}\n")
            if reports:
                for role, content in reports.items():
                    f.write(f"\n--- {role.upper()} ---\n{content}\n")
            elif single_text:
                f.write(single_text + "\n")

    # Pipeline execution
    selection_data = await run_round_0_selection(paper_file, N=N_personas)
    active_personas = selection_data["selected_personas"]
    log_stage(f"ROUND 0: SELECTION (N={N_personas})", single_text=json.dumps(selection_data, indent=2))

    r1_reports = await run_round_1_serial(active_personas, paper_file)
    log_stage("ROUND 1: INDEPENDENT EVALUATION", reports=r1_reports)

    r2a_reports = await run_round_2a_serial(r1_reports, active_personas, paper_file)
    log_stage("ROUND 2A: CROSS-EXAMINATION", reports=r2a_reports)

    r2b_reports = await run_round_2b_serial(r2a_reports, active_personas, paper_file)
    log_stage("ROUND 2B: ANSWERING CLARIFICATIONS", reports=r2b_reports)

    r2c_reports = await run_round_2c_serial(r1_reports, r2a_reports, r2b_reports, active_personas, paper_file)
    log_stage("ROUND 2C: FINAL AMENDMENTS", reports=r2c_reports)

    final_decision = await run_round_3(r2c_reports, selection_data)
    log_stage("FINAL REFEREE REPORT (EDITOR)", single_text=final_decision)

    print(f"\n[SUCCESS] Experiment N={N_personas} completed!")
    return final_decision

In [ ]:
# Set the number of personas you want the LLM to endogenously select
TARGET_N = 3

# Ensure paper1 is defined from your earlier upload block
print(f"Executing pipeline with {TARGET_N} dynamically selected personas...")

# Fire the orchestrator
final_output = await execute_experiment(paper1, N_personas=TARGET_N)

print("\n" + "="*60)
print(f"FINAL OUTPUT FOR N={TARGET_N}")
print("="*60)
print(final_output)

Executing pipeline with 3 dynamically selected personas...

[SYSTEM] Saving outputs to: /content/drive/MyDrive/SENIOR_YEAR/FED/Experiment_N3/Review_N3_20260415_200755.txt

--- STARTING ROUND 0: ENDOGENOUS SELECTION (N=3) ---
[Editor_Selector] Generating response using gemini-2.5-flash...

[🚨 QUOTA] gemini-2.5-flash exhausted. Rerouting to gemini-2.5-pro...
[Editor_Selector] Generating response using gemini-2.5-pro...

[🚨 QUOTA] gemini-2.5-pro exhausted. Rerouting to gemini-2.0-flash...
[Editor_Selector] Generating response using gemini-2.0-flash...

[🚨 QUOTA] gemini-2.0-flash exhausted. Rerouting to gemini-2.0-flash-001...
[Editor_Selector] Generating response using gemini-2.0-flash-001...

[🚨 QUOTA] gemini-2.0-flash-001 exhausted. Rerouting to gemini-2.0-flash-lite-001...
[Editor_Selector] Generating response using gemini-2.0-flash-lite-001...

[🚨 QUOTA] gemini-2.0-flash-lite-001 exhausted. Rerouting to gemini-2.0-flash-lite...
[Editor_Selector] Generating response using gemini-2.0-fl

## Aside: consistency tester

In [ ]:
import pandas as pd
import numpy as np
import random
import time

async def run_weight_consistency_experiment(paper_file, N_personas, iterations=10):
    print(f"\n--- RUNNING EDITOR CONSISTENCY TEST (N={N_personas}, {iterations} Iterations) ---")

    results = []
    all_personas = list(SYSTEM_PROMPTS.keys())

    for i in range(iterations):
        print(f"Executing Iteration {i+1}/{iterations}...")

        # ---------------------------------------------------------
        # 1. ENDOGENOUS SELECTION (The LLM Editor)
        # ---------------------------------------------------------
        try:
            # We call only the Round 0 function
            selection_data = await run_round_0_selection(paper_file, N=N_personas)
            endogenous_personas = selection_data.get("selected_personas", [])
            endogenous_weights = selection_data.get("weights", {})

            # Sort them alphabetically for easy comparison in the dataframe
            endogenous_panel = ", ".join(sorted(endogenous_personas))

        except Exception as e:
            print(f"Error on iteration {i+1}: {e}")
            endogenous_panel = "ERROR"
            endogenous_weights = "ERROR"

        # ---------------------------------------------------------
        # 2. RANDOM BASELINE GENERATION
        # ---------------------------------------------------------
        # Pick N random personas from the pool of 10
        random_personas = random.sample(all_personas, N_personas)

        # Generate random weights that perfectly sum to 1.0
        random_raw = np.random.dirichlet(np.ones(N_personas), size=1)[0]
        random_weights = {role: round(weight, 3) for role, weight in zip(random_personas, random_raw)}

        random_panel = ", ".join(sorted(random_personas))

        # ---------------------------------------------------------
        # 3. LOGGING THE DATA
        # ---------------------------------------------------------
        results.append({
            "Iteration": i + 1,
            "Endogenous_Panel": endogenous_panel,
            "Endogenous_Weights": str(endogenous_weights),
            "Random_Panel": random_panel,
            "Random_Weights": str(random_weights)
        })

        # A short 4-second sleep to ensure we don't trip the 15 RPM limit
        # since these Round 0 calls are very fast.
        await asyncio.sleep(4)

    # Convert results to a Pandas DataFrame for easy analysis
    df = pd.DataFrame(results)
    print("\n--- EXPERIMENT COMPLETE ---")
    return df

In [ ]:
# Make sure your paper1 variable is still active in memory!
consistency_df = await run_weight_consistency_experiment(paper1, N_personas=3, iterations=10)

# Display the resulting dataframe
display(consistency_df)


--- RUNNING EDITOR CONSISTENCY TEST (N=3, 10 Iterations) ---
Executing Iteration 1/10...

--- STARTING ROUND 0: ENDOGENOUS SELECTION (N=3) ---
[Editor_Selector] Generating response using gemini-2.5-flash...
[SYSTEM] Selected Panel: ['Econometrician', 'ML_Expert', 'Policymaker']
Executing Iteration 2/10...

--- STARTING ROUND 0: ENDOGENOUS SELECTION (N=3) ---
[Editor_Selector] Generating response using gemini-2.5-flash...
[SYSTEM] Selected Panel: ['Econometrician', 'ML_Expert', 'Policymaker']
Executing Iteration 3/10...

--- STARTING ROUND 0: ENDOGENOUS SELECTION (N=3) ---
[Editor_Selector] Generating response using gemini-2.5-flash...
[SYSTEM] Selected Panel: ['Econometrician', 'ML_Expert', 'Policymaker']
Executing Iteration 4/10...

--- STARTING ROUND 0: ENDOGENOUS SELECTION (N=3) ---
[Editor_Selector] Generating response using gemini-2.5-flash...
[SYSTEM] Selected Panel: ['Econometrician', 'ML_Expert', 'Policymaker']
Executing Iteration 5/10...

--- STARTING ROUND 0: ENDOGENOUS SELE

,Iteration,Endogenous_Panel,Endogenous_Weights,Random_Panel,Random_Weights
0,1,"Econometrician, ML_Expert, Policymaker","{'Econometrician': 0.4, 'ML_Expert': 0.35, 'Po...","Econometrician, Theorist, Visionary","{'Econometrician': np.float64(0.112), 'Theoris..."
1,2,"Econometrician, ML_Expert, Policymaker","{'Econometrician': 0.4, 'ML_Expert': 0.35, 'Po...","CS_Expert, Econometrician, Visionary","{'CS_Expert': np.float64(0.073), 'Econometrici..."
2,3,"Econometrician, ML_Expert, Policymaker","{'Econometrician': 0.4, 'ML_Expert': 0.4, 'Pol...","Econometrician, ML_Expert, Policymaker","{'ML_Expert': np.float64(0.558), 'Policymaker'..."
3,4,"Econometrician, ML_Expert, Policymaker","{'Econometrician': 0.4, 'ML_Expert': 0.35, 'Po...","Data_Scientist, Ethicist, Perspective","{'Ethicist': np.float64(0.056), 'Data_Scientis..."
4,5,"Econometrician, ML_Expert, Policymaker","{'Econometrician': 0.4, 'ML_Expert': 0.3, 'Pol...","ML_Expert, Perspective, Theorist","{'ML_Expert': np.float64(0.566), 'Perspective'..."
5,6,"Econometrician, ML_Expert, Policymaker","{'Econometrician': 0.4, 'ML_Expert': 0.35, 'Po...","Historian, Perspective, Theorist","{'Historian': np.float64(0.173), 'Perspective'..."
6,7,"Econometrician, ML_Expert, Policymaker","{'ML_Expert': 0.4, 'Econometrician': 0.4, 'Pol...","CS_Expert, Data_Scientist, Visionary","{'Visionary': np.float64(0.149), 'Data_Scienti..."
7,8,"Econometrician, ML_Expert, Policymaker","{'Econometrician': 0.4, 'ML_Expert': 0.4, 'Pol...","CS_Expert, Policymaker, Theorist","{'Theorist': np.float64(0.005), 'CS_Expert': n..."
8,9,"Econometrician, ML_Expert, Policymaker","{'Econometrician': 0.4, 'ML_Expert': 0.35, 'Po...","CS_Expert, Ethicist, Historian","{'CS_Expert': np.float64(0.18), 'Historian': n..."
9,10,"Econometrician, ML_Expert, Policymaker","{'Econometrician': 0.4, 'ML_Expert': 0.35, 'Po...","Data_Scientist, Ethicist, Historian","{'Data_Scientist': np.float64(0.085), 'Ethicis..."
